# Notebook 03 — SageMaker Pipeline with Post-Pipeline SageMaker MLflow App Logging

**Module:** ITI113 Machine Learning & Operations  
**Focus Area:** C — MLOps (Pipeline, experiment tracking, CI/CD, deployment)  
**Estimated Runtime:** 15–25 minutes for the pipeline execution

---

## What this notebook does

1. Writes `preprocess.py`, `train.py`, and `inference.py` to a local `src/` folder.
2. Defines and runs a SageMaker Pipeline: **Process → Train → Condition → Register**.
3. Keeps the SageMaker training container free of MLflow credentials and MLflow dependencies.
4. After a successful pipeline run, the notebook reads SageMaker job metadata and metrics, then logs them to the team’s **SageMaker Serverless MLflow App**. The notebook validates the app `TeamId` tag before logging.
5. Registers a quality-approved model in **SageMaker Model Registry**.

6. Saves a `joblib` deployment bundle containing the trained model and preprocessing artefacts, so the endpoint can accept raw 13-feature JSON input.

This separation makes troubleshooting safer and simpler:

```text
SageMaker Pipeline                    Notebook-side MLflow logging
Process → Train → Gate → Register     Read run metadata → Log to MLflow App
```

This version follows Notebook 01/02 and uses the SageMaker MLflow App ARN when available from `mlflow_app_config_team01_s004.json`.


> **Adapted for team03 (Ong Hui Lin, Student 2 — MLOps & Deployment) from the ITI113 course template notebook `03_sagemaker_pipeline_mlflow_app_with_preprocessing_bundle.ipynb`.** The pipeline shape (ProcessingStep → TrainingStep → ModelStep/Register → ConditionStep quality gate → Serverless Endpoint deploy), the post-pipeline SageMaker MLflow App logging, and the team-tag safety check are all unchanged from the tutor's original. What changed, because this project is text classification (TF-IDF) rather than tabular heart-disease data, is marked **`ADAPTED`** below: `preprocess.py`/`train.py`/`inference.py` do text cleaning + engineered indicator features + TF-IDF instead of imputation/scaling, the deployment bundle carries a fitted TF-IDF vectorizer instead of a `StandardScaler`, the endpoint accepts raw message text (`{"text": "..."}`) instead of 13 numeric fields, and the quality gate is 0.85 ROC-AUC (matching the project proposal and Notebook 02's `QUALITY_GATE`) instead of the tutor's 0.75 example. Random Forest hyperparameters (`n_estimators=200, max_depth=8, min_samples_leaf=2`) default to `rf_candidate_05`, the winning run from Notebook 02's hyperparameter tuning (`best_model.json`).
>
> **GitHub Actions CI/CD section is skipped**, per the tutor's own `[SKIP THIS]` marking on that section.

In [1]:
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow


Note: you may need to restart the kernel to use updated packages.


## 0. Configuration

In [2]:
import boto3
import sagemaker
import json
import os
import time
from pathlib import Path

# ----------------------------
# AWS / SageMaker setup
# ----------------------------
session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

BUCKET  = "nyp-26s1-iti113"

TEAM_ID = "team03"
STUDENT_ID = "s301"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "crypto-scam-detector"

PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

# ----------------------------
# SageMaker Serverless MLflow App setup
# ----------------------------
TEAM_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

config_candidates = [
    TEAM_CONFIG_FILE,
    STUDENT_CONFIG_FILE,
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]

MLFLOW_APP_ARN = None
MLFLOW_EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment1"
mlflow_config = {}
config_used = None

for config_file in config_candidates:
    if config_file.exists():
        mlflow_config = json.loads(config_file.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = (
            mlflow_config.get("MLFLOW_APP_ARN")
            or mlflow_config.get("mlflow_app_arn")
            or mlflow_config.get("arn")
        )
        MLFLOW_EXPERIMENT_NAME = (
            mlflow_config.get("EXPERIMENT_NAME")
            or mlflow_config.get("experiment_name")
            or MLFLOW_EXPERIMENT_NAME
        )
        config_used = config_file
        break

# Fallback for classroom testing only -- team03's MLflow App ARN from Notebook 01A.
DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-J5AYUG4AJHVW"
)

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print(
        "[WARNING] No local MLflow config file found. "
        "Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team."
    )
else:
    print(f"Loaded MLflow App config from {config_used}")

config_team_id = mlflow_config.get("TEAM_ID") or mlflow_config.get("team_id")
if config_team_id and config_team_id != TEAM_ID:
    raise ValueError(
        f"Config file team mismatch: config TEAM_ID={config_team_id}, notebook TEAM_ID={TEAM_ID}. "
        "Do not use another team's MLflow config."
    )

# ----------------------------
# Safety check for team-level MLflow restriction
# ----------------------------
sm_for_mlflow = boto3.client("sagemaker", region_name=region)

try:
    tag_response = sm_for_mlflow.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}

    print("MLflow App tags:")
    for k, v in mlflow_app_tags.items():
        print(f"  {k}: {v}")

    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App."
        )

    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")

except Exception as e:
    print("\n[ERROR] Could not validate MLflow App team tag.")
    print("This usually means one of the following:")
    print("1. The MLflow App ARN belongs to another team and IAM correctly blocked access.")
    print("2. The MLflow App is missing the TeamId tag.")
    print("3. The current role lacks permission to list tags for this MLflow App.")
    print(type(e).__name__, e)
    raise

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_APP_ARN
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME


def create_mlflow_app_presigned_url(fragment: str = "") -> str:
    sm_for_mlflow = boto3.client("sagemaker", region_name=region)
    response = sm_for_mlflow.create_presigned_mlflow_app_url(
        Arn=MLFLOW_APP_ARN
    )

    base_url = response.get("AuthorizedUrl") or response.get("Url")

    if not base_url:
        raise RuntimeError(
            "create_presigned_mlflow_app_url did not return AuthorizedUrl or Url. "
            f"Response: {response}"
        )

    base_url = base_url.split("#", 1)[0]

    if fragment:
        return base_url + "#" + fragment.lstrip("#")

    return base_url


def print_mlflow_presigned_links(experiment_id=None, run_id=None):
    if experiment_id is not None:
        experiment_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}"
        )
        print("Presigned MLflow experiment URL:")
        print(experiment_url)

    if experiment_id is not None and run_id is not None:
        run_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}/runs/{run_id}"
        )
        print("\nPresigned MLflow run URL:")
        print(run_url)

PIPELINE_NAME       = f"iti113-{TEAM_ID}-crypto-scam-detector"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-CryptoScamDetector"
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-crypto-scam-detector"
# ADAPTED: matches the project proposal's ROC-AUC threshold (also used in Notebook 02's
# QUALITY_GATE), not the tutor's 0.75 example value.
QUALITY_GATE_AUC    = 0.85

RAW_DATA_URI  = f"s3://{BUCKET}/{PREFIX}/raw/crypto_scam_dataset.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"

SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI    = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

print(f"Pipeline                : {PIPELINE_NAME}")
print(f"Bucket                  : {BUCKET}")
print(f"Team prefix             : {PREFIX}")
print(f"Semester                : {SEMESTER}")
print(f"Region                  : {region}")
print(f"SageMaker role          : {role}")
print(f"MLflow App ARN          : {MLFLOW_APP_ARN}")
print(f"MLflow experiment       : {MLFLOW_EXPERIMENT_NAME}")
print(f"Raw data URI            : {RAW_DATA_URI}")
print(f"Pipeline source S3 URI  : {SCRIPTS_S3_URI}")
print(f"Local pipeline source   : {LOCAL_PIPELINE_SRC}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Loaded MLflow App config from mlflow_app_config_team03_s301.json


MLflow App tags:
  Semester: 26S1
  sagemaker:domain-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-gpdrdk2w4dgw
  ProjectName: crypto-scam-detector
  sagemaker:space-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-gpdrdk2w4dgw/team03-shared
  Course: ITI113
  TeamId: team03
  CreatedByNotebook: 01A_setup_sagemaker_mlflow_app
  StudentId: s301
[OK] MLflow App tag TeamId=team03 matches notebook TEAM_ID=team03
Pipeline                : iti113-team03-crypto-scam-detector
Bucket                  : nyp-26s1-iti113
Team prefix             : iti113/team03/data/crypto-scam-detector
Semester                : 26S1
Region                  : ap-southeast-1
SageMaker role          : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team03
MLflow App ARN          : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
MLflow experiment       : ITI113/team03/Experiment1
Raw data URI            : s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam

## 0A. Precheck SageMaker MLflow App connection

Before launching the SageMaker Pipeline, test that this notebook can create/use the **current team's** MLflow experiment in the SageMaker MLflow App.

Example for Team 40:

```text
ITI113/team40/Experiment1
```

This notebook now checks that the selected MLflow App has the correct `TeamId` tag before logging. If this fails, resolve the MLflow App ARN, `sagemaker-mlflow` package, or IAM permissions before continuing to the SageMaker Pipeline.


### Note about MLflow links

The MLflow client may print links such as `https://mlflow.sagemaker.ap-southeast-1.app.aws/#/...`. Those generic links are not presigned and may show a SageMaker MLflow permission/session error. This notebook generates fresh presigned MLflow App URLs using `create_presigned_mlflow_app_url()` after each logging step. Use those printed presigned URLs instead.


In [3]:
import mlflow
import time

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{TEAM_ID}_pipeline_notebook_precheck_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "crypto-scam-detector",
        "run_type": "sagemaker_pipeline_precheck",
        "tracking_backend": "sagemaker_mlflow_app",
        "mlflow_app_arn": MLFLOW_APP_ARN,
    })
    mlflow.log_param("source", "notebook_03_precheck")
    mlflow.log_metric("connection_success", 1)

    precheck_run_id = run.info.run_id
    precheck_experiment_id = run.info.experiment_id

print("SageMaker MLflow App precheck completed.")
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", precheck_experiment_id)
print("Run ID:", precheck_run_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=precheck_experiment_id,
    run_id=precheck_run_id
)


🏃 View run team03_pipeline_notebook_precheck_1785566388 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/5a47c8ed588c4d5aafedec2c74afbf1d
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App precheck completed.
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
Experiment: ITI113/team03/Experiment1
Experiment ID: 1
Run ID: 5a47c8ed588c4d5aafedec2c74afbf1d

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-J5AYUG4AJHVW.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlJZT0NNNiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNE14Z29GK00xdUxZRVNia2ZRN2sxb0U1RC9HSWIyZlZWUm1xOEtoM0diWW9BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFdmMyNTNUbTlTY20xVlRYVlVZV3RuUWtoaU1EYzVTWEZ0Um1SdlNFWnFTSFkxUVdsMVZrNTVXV0pXVEdWNlRqWkRjeTlzYlVocU1WUXdLemhGZVdscmR6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFheFNtRUdSbjJ1RE54akhVS01sV0ZVQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF3V0hwb1VNcDB1ZC9vZVBEQUNBUkNBTzFOaGUrd0Y3dndwUWFTcVZ5TUFpSnoyNTNma000VXNYYWdCNng2cjZTMGoyZlVKaGF2U0VCOFlzUkVpL0lCMzhvaXkzZ0RrNjFnSFZGeW5BZ0FBRUFEMEpOTmVDZDNwamc2aFo2bXRxaDNYekoyS3F2MnFsVjAzRyszS1BZWjE4UjZuSzExdVU2eG


Presigned MLflow run URL:
https://app-J5AYUG4AJHVW.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkFLTkVQSiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDNBOUx6dlNzU2ZFZFpadWxTSWpyTFpjYjQxL1paOFV3SVpLa2x5SWRzN29BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFMFJESnhiMlphTTNKVFNWZFphVlJOVHpaTlZHeEpSMVUxUjBoYVJHMVVlUzgzT0RaS05tbE9WR001WWxCQldERk9iMVZIV25sME1TdE9ORWhUZUVZMmR6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFmUENMU3Q5d3RMR0ROa09PWjl0TlJ3QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF5aEQwVWVMQ3NJaGJLU1NyUUNBUkNBT3lrejFCWWFkQ3NlRVYwSTJUd0pnQkpmRnNWbGRlUHVGV0xOWWNOUHhCb2tCT2dxMEx3U0NLaTdmL1dKcmdOaFZyWkpEUGo3dlYrN0M3KzdBZ0FBRUFCbGdEaW9tWTB4dU5RMVY0S05mVnlaTjFoM0dqaE1MMXVHZW82WmZ5TkdGVVFicDZKZlVRNk1CTDFs

## 1. Write Pipeline Scripts

SageMaker Pipeline steps run as isolated jobs, each in its own managed container.

This updated version saves a **deployment bundle** using `joblib`. The bundle contains:

- the trained Random Forest model,
- the fitted `StandardScaler`,
- the training medians and modes used for missing-value handling,
- the raw input column list, and
- the final processed feature column order.

This means the deployed endpoint can accept the more conventional **13 raw Heart Disease input fields** and perform lightweight preprocessing inside `inference.py` before prediction.

The SageMaker Processing step is still used for training-time preprocessing. However, for live endpoint prediction, no separate Processing job is started. The endpoint performs the required single-row preprocessing in memory.

MLflow logging is intentionally performed outside the SageMaker training container by the notebook after a successful pipeline execution.


In [4]:
os.makedirs('src', exist_ok=True)
print('src/ directory ready')


src/ directory ready


In [ ]:
%%writefile src/preprocess.py
"""SageMaker Processing Job -- replicates Notebooks 01/02 preprocessing for text data.

Cleans the raw scam-message text, builds the engineered indicator features
(urgency, contact/link, structural characteristics), fits TF-IDF on the
training split only, and saves everything the training job and the deployed
endpoint need to stay in sync: TF-IDF features (sparse .npz), engineered
features and labels (CSV), and a preprocessor bundle (joblib) containing the
fitted vectorizer.
"""
import os
import re
import argparse
import glob

import pandas as pd
import numpy as np
import joblib
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

parser = argparse.ArgumentParser()
parser.add_argument('--test-size',    type=float, default=0.20)
parser.add_argument('--random-state', type=int,   default=42)
args = parser.parse_args()

# BUGFIX (Progress Check debugging, 1 Aug 2026): this used to be hardcoded to
# 'crypto_scam_dataset.csv', which only matched the manual pipeline's fixed raw
# file. Notebook 05's whole point is that InputDataUrl -- and therefore the
# downloaded filename -- changes with whatever CSV triggered the run, so the
# hardcoded name caused a FileNotFoundError ("AlgorithmError, exit code: 1")
# the first time a differently-named file (synthetic_batch_manual-test.csv)
# triggered this script. Discover the file instead of assuming its name.
input_dir = '/opt/ml/processing/input'
csv_candidates = sorted(glob.glob(os.path.join(input_dir, '*.csv')))
if not csv_candidates:
    raise FileNotFoundError(
        f'No CSV file found in {input_dir}. Expected the file that triggered '
        'this pipeline run (or crypto_scam_dataset.csv for a manual run).'
    )
input_path = csv_candidates[0]
print(f'Using input file: {input_path}')
output_dir = '/opt/ml/processing/output'
os.makedirs(output_dir, exist_ok=True)

SCAM_LABEL = "scam"

# ---------------------------------------------------------------------------
# Text cleaning (mirrors utils/preprocessing.py's clean_text())
# ---------------------------------------------------------------------------
URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ---------------------------------------------------------------------------
# Engineered indicator features (mirrors utils/indicators.py + utils/preprocessing.py)
# ---------------------------------------------------------------------------
URGENT_KEYWORDS = [
    "urgent", "immediately", "act now", "act fast", "hurry", "limited time",
    "today only", "expires today", "offer ends soon", "last chance",
    "don't miss out", "within 24 hours", "within 1 hour", "respond now",
    "claim now", "limited slots", "before it's too late", "time-sensitive",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed return", "guaranteed returns", "guaranteed profit",
    "risk-free", "risk free", "100% profit", "double your money",
    "high returns", "\u7a33\u8d5a\u4e0d\u8d54",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer funds", "send payment", "pay now", "top up",
    "bitcoin", "btc", "ethereum", "eth", "usdt", "wallet address",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "wechat", "private chat",
    "dm me", "direct message",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "wallet password", "recovery phrase",
    "otp", "verification code", "security code",
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches(message, keywords):
    message_lower = message.lower()
    return [k for k in keywords if k.lower() in message_lower]

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "wallet_address_count": len(wallet_matches),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "message_length": len(text),
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
        "digit_count": digit_count,
    }

ENGINEERED_COLS = [
    "urgency_keyword_count", "guaranteed_return_keyword_count", "countdown_phrase_count",
    "exclamation_count", "urgency_score", "has_wallet_address", "wallet_address_count",
    "has_url", "url_count", "has_email", "has_phone_number", "payment_keyword_count",
    "off_platform_keyword_count", "credential_keyword_count", "message_length",
    "capital_letter_ratio", "has_numeric_content", "digit_count",
]

# ---------------------------------------------------------------------------
# Load, clean, engineer, split
# ---------------------------------------------------------------------------
df = pd.read_csv(input_path)
expected_columns = {"id", "platform", "text", "label"}
missing = expected_columns - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing expected columns: {missing}")

df["clean_text"] = df["text"].apply(clean_text)

engineered = pd.DataFrame([extract_engineered_features(t) for t in df["text"]], index=df.index)
df = pd.concat([df, engineered], axis=1)

train_df, test_df = train_test_split(
    df,
    test_size=args.test_size,
    random_state=args.random_state,
    stratify=df["label"],
)

print(f"Train: {train_df.shape[0]} rows | Test: {test_df.shape[0]} rows")

# ---------------------------------------------------------------------------
# TF-IDF -- fit on TRAIN text only, never on test
# ---------------------------------------------------------------------------
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
vectorizer.fit(train_df["clean_text"])

X_train_tfidf = vectorizer.transform(train_df["clean_text"])
X_test_tfidf = vectorizer.transform(test_df["clean_text"])

y_train = (train_df["label"] == SCAM_LABEL).astype(int)
y_test = (test_df["label"] == SCAM_LABEL).astype(int)

print(f"TF-IDF vocabulary size: {len(vectorizer.get_feature_names_out())}")

# ---------------------------------------------------------------------------
# Save outputs -- sparse TF-IDF as .npz (dense CSV would be hundreds of MB
# for a 5000-column matrix), everything else as CSV.
# ---------------------------------------------------------------------------
sp.save_npz(f"{output_dir}/train_tfidf.npz", X_train_tfidf)
sp.save_npz(f"{output_dir}/test_tfidf.npz", X_test_tfidf)

train_df[ENGINEERED_COLS].to_csv(f"{output_dir}/train_engineered.csv", index=False)
test_df[ENGINEERED_COLS].to_csv(f"{output_dir}/test_engineered.csv", index=False)

y_train.to_csv(f"{output_dir}/train_labels.csv", index=False, header=True)
y_test.to_csv(f"{output_dir}/test_labels.csv", index=False, header=True)

preprocessor = {
    "vectorizer": vectorizer,
    "engineered_cols": ENGINEERED_COLS,
    "feature_columns": list(vectorizer.get_feature_names_out()) + ENGINEERED_COLS,
}
joblib.dump(preprocessor, f"{output_dir}/preprocessor.joblib")

print("Preprocessing complete. Saved TF-IDF (.npz), engineered features, labels, and preprocessor.joblib.")
print(f"Final feature count: {len(preprocessor['feature_columns'])}")

In [6]:
%%writefile src/train.py
"""
SageMaker Training Job
-----------------------
Trains a Random Forest classifier on TF-IDF + engineered features and saves
a deployment bundle.

Hyperparameters default to the winning configuration from Notebook 02's
hyperparameter tuning (rf_candidate_05: n_estimators=200, max_depth=8,
min_samples_leaf=2, test AUC-ROC from that run), overridable as SageMaker
Pipeline parameters.

The deployment bundle contains the trained model AND the fitted TF-IDF
vectorizer (from the Processing step's preprocessor.joblib), so the endpoint
can accept raw message text and reproduce training-time feature engineering.

MLflow logging is intentionally performed outside the SageMaker training
container by the notebook after a successful pipeline execution.
"""
import os
import argparse
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
)

parser = argparse.ArgumentParser()
parser.add_argument('--n-estimators', type=int, default=200)
parser.add_argument('--max-depth', type=int, default=8)
parser.add_argument('--min-samples-leaf', type=int, default=2)
parser.add_argument('--random-state', type=int, default=42)

parser.add_argument('--team-id', type=str, default=os.environ.get('TEAM_ID', 'unknown-team'))
parser.add_argument('--student-id', type=str, default=os.environ.get('STUDENT_ID', 's000'))
parser.add_argument('--semester', type=str, default=os.environ.get('SEMESTER', '26S1'))
parser.add_argument('--run-name', type=str, default='sagemaker_pipeline_run')

parser.add_argument(
    '--model-dir',
    type=str,
    default=os.environ.get('SM_MODEL_DIR', '/opt/ml/model')
)
parser.add_argument(
    '--train',
    type=str,
    default=os.environ.get('SM_CHANNEL_TRAIN', '/opt/ml/input/data/train')
)
parser.add_argument(
    '--test',
    type=str,
    default=os.environ.get('SM_CHANNEL_TEST', '/opt/ml/input/data/test')
)
args = parser.parse_args()

os.makedirs(args.model_dir, exist_ok=True)

print('=== SageMaker Training Environment ===')
print(f'Train channel: {args.train}')
print(f'Test channel: {args.test}')
print(f'Model directory: {args.model_dir}')

X_train_tfidf = sp.load_npz(os.path.join(args.train, 'train_tfidf.npz'))
X_test_tfidf = sp.load_npz(os.path.join(args.test, 'test_tfidf.npz'))

train_engineered = pd.read_csv(os.path.join(args.train, 'train_engineered.csv'))
test_engineered = pd.read_csv(os.path.join(args.test, 'test_engineered.csv'))

y_train = pd.read_csv(os.path.join(args.train, 'train_labels.csv')).squeeze('columns')
y_test = pd.read_csv(os.path.join(args.test, 'test_labels.csv')).squeeze('columns')

preprocessor_path = os.path.join(args.train, 'preprocessor.joblib')
if not os.path.exists(preprocessor_path):
    raise FileNotFoundError(
        f'preprocessor.joblib was not found at {preprocessor_path}. '
        'Rerun the ProcessingStep with the updated preprocess.py.'
    )

preprocessor = joblib.load(preprocessor_path)
engineered_cols = preprocessor['engineered_cols']

X_train = sp.hstack([X_train_tfidf, train_engineered[engineered_cols].values]).tocsr()
X_test = sp.hstack([X_test_tfidf, test_engineered[engineered_cols].values]).tocsr()

print(f'Train: {X_train.shape[0]} rows, {X_train.shape[1]} features')
print(f'Test : {X_test.shape[0]}  rows')

if len(pd.Series(y_train).unique()) < 2:
    raise ValueError('Training labels contain fewer than two classes.')

model = RandomForestClassifier(
    n_estimators=args.n_estimators,
    max_depth=args.max_depth,
    min_samples_leaf=args.min_samples_leaf,
    class_weight='balanced',
    random_state=args.random_state,
    n_jobs=-1,
)
model.fit(X_train, y_train)

all_metrics = {}
for split, X, y in [('train', X_train, y_train), ('test', X_test, y_test)]:
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    all_metrics.update({
        f'{split}_accuracy': round(accuracy_score(y, predictions), 4),
        f'{split}_f1': round(f1_score(y, predictions, zero_division=0), 4),
        f'{split}_precision': round(
            precision_score(y, predictions, zero_division=0), 4
        ),
        f'{split}_recall': round(
            recall_score(y, predictions, zero_division=0), 4
        ),
    })

    if len(pd.Series(y).unique()) >= 2:
        all_metrics[f'{split}_auc_roc'] = round(
            roc_auc_score(y, probabilities), 4
        )
    else:
        all_metrics[f'{split}_auc_roc'] = None
        print(f'Warning: {split} split has only one class; AUC-ROC unavailable.')

print('=== Metrics ===')
for metric_name, metric_value in all_metrics.items():
    print(f'{metric_name}: {metric_value}')

model_bundle = {
    'model': model,
    'preprocessor': preprocessor,
    'engineered_cols': engineered_cols,
    'input_format': 'raw_text_json',
    'description': (
        'Deployment bundle containing trained model and fitted TF-IDF vectorizer. '
        'Endpoint accepts raw message text as JSON: {"text": "..."}'
    ),
}

joblib.dump(model_bundle, os.path.join(args.model_dir, 'model.joblib'))

with open(os.path.join(args.model_dir, 'model.pkl'), 'wb') as f:
    pickle.dump(model_bundle, f)

print(f"Model bundle saved: {os.path.join(args.model_dir, 'model.joblib')}")
print(f"Legacy bundle saved: {os.path.join(args.model_dir, 'model.pkl')}")

if all_metrics['test_auc_roc'] is None:
    raise ValueError('Test AUC-ROC is unavailable; cannot evaluate the quality gate.')

print(f"Test AUC-ROC: {all_metrics['test_auc_roc']}")
print(f"test_accuracy: {all_metrics['test_accuracy']}")
print(f"test_f1: {all_metrics['test_f1']}")

Writing src/train.py


In [7]:
%%writefile src/inference.py
"""SageMaker inference handler for the deployed crypto-scam-detector endpoint.

Accepts JSON input with raw message text:

{"text": "Deposit 500 USDT today and receive guaranteed returns..."}

or a list of such records, or {"instances": [...]}. Applies the same text
cleaning, engineered-feature extraction, and TF-IDF transform saved in
model.joblib before calling the trained model -- the same logic used in
Notebooks 01/02 and utils/preprocessing.py, so serving matches training.
"""
import os
import re
import json
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

URGENT_KEYWORDS = [
    "urgent", "immediately", "act now", "act fast", "hurry", "limited time",
    "today only", "expires today", "offer ends soon", "last chance",
    "don't miss out", "within 24 hours", "within 1 hour", "respond now",
    "claim now", "limited slots", "before it's too late", "time-sensitive",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed return", "guaranteed returns", "guaranteed profit",
    "risk-free", "risk free", "100% profit", "double your money",
    "high returns", "\u7a33\u8d5a\u4e0d\u8d54",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer funds", "send payment", "pay now", "top up",
    "bitcoin", "btc", "ethereum", "eth", "usdt", "wallet address",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "wechat", "private chat",
    "dm me", "direct message",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "wallet password", "recovery phrase",
    "otp", "verification code", "security code",
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches(message, keywords):
    message_lower = message.lower()
    return [k for k in keywords if k.lower() in message_lower]

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "wallet_address_count": len(wallet_matches),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "message_length": len(text),
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
        "digit_count": digit_count,
    }


def model_fn(model_dir):
    """Load the model bundle from the SageMaker model directory."""
    joblib_path = os.path.join(model_dir, 'model.joblib')
    pkl_path = os.path.join(model_dir, 'model.pkl')

    if os.path.exists(joblib_path):
        bundle = joblib.load(joblib_path)
    elif os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            bundle = pickle.load(f)
    else:
        raise FileNotFoundError('Neither model.joblib nor model.pkl was found.')

    return bundle


def input_fn(body, content_type='application/json'):
    """Parse JSON request body into a DataFrame of raw message-text records."""
    if content_type != 'application/json':
        raise ValueError(f'Unsupported content type: {content_type}')

    payload = json.loads(body)

    if isinstance(payload, dict):
        if 'instances' in payload:
            payload = payload['instances']
        else:
            payload = [payload]

    if not isinstance(payload, list):
        raise ValueError('JSON input must be a dictionary, a list of dictionaries, or {"instances": [...]}')

    return pd.DataFrame(payload)


def predict_fn(data, bundle):
    """Clean text, extract engineered features, TF-IDF transform, then predict."""
    if 'text' not in data.columns:
        raise ValueError('Input must include a "text" field with the raw message.')

    model = bundle['model']
    preprocessor = bundle['preprocessor']
    vectorizer = preprocessor['vectorizer']
    engineered_cols = preprocessor['engineered_cols']

    clean = data['text'].apply(clean_text)
    engineered = pd.DataFrame(
        [extract_engineered_features(t) for t in data['text']], index=data.index
    )

    X_tfidf = vectorizer.transform(clean)
    X = sp.hstack([X_tfidf, engineered[engineered_cols].values]).tocsr()

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    return predictions, probabilities


def output_fn(prediction, accept='application/json'):
    preds, probas = prediction
    response = [
        {
            'prediction': int(p),
            'label': 'Scam' if int(p) == 1 else 'Legitimate',
            'probability': round(float(b), 4),
        }
        for p, b in zip(preds, probas)
    ]
    return json.dumps(response), accept

Writing src/inference.py


In [8]:
# No MLflow requirements file is needed in the SageMaker training container.
# MLflow logging is done after the pipeline completes, from this notebook.
# The preprocessing script now saves a joblib preprocessor bundle containing:
#   - fitted TF-IDF vectorizer
#   - engineered feature column names
#   - full feature column order (TF-IDF vocab + engineered columns)
# The training script bundles this preprocessor together with the trained
# model into model.joblib, so the endpoint can accept raw message text.

print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")

Scripts written:
  src/preprocess.py  (8234 bytes)
  src/train.py  (5701 bytes)
  src/inference.py  (6573 bytes)


## 1A. Upload pipeline source files to S3

This section pre-places the three pipeline source files in the team S3 area.

Files uploaded:

```text
preprocess.py
train.py
inference.py
```

There is no `requirements_train.txt`: the SageMaker training container does not need MLflow or MLflow App packages. MLflow logging happens later from this notebook to the SageMaker MLflow App.

**ADAPTED:** the tutor's `preprocess.py`/`train.py` save a fitted `StandardScaler` and imputation values for tabular data; this project's version saves a fitted TF-IDF vectorizer and the engineered indicator feature columns instead. The deployed endpoint accepts raw message-text JSON input (`{"text": "..."}`) and performs the same cleaning, feature engineering, and TF-IDF transform inside `inference.py` before predicting.

S3 location:

```text
s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/
```

In [9]:
from pathlib import Path

s3_client = boto3.client("s3")

SOURCE_DIR = Path("src")
FILES_TO_UPLOAD = [
    "preprocess.py",
    "train.py",
    "inference.py",
]

for filename in FILES_TO_UPLOAD:
    local_path = SOURCE_DIR / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)

Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py


Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py


Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src


## 1B. Download pipeline source files from S3

The SageMaker Pipeline below will use the local `pipeline_src/` folder, but that folder is recreated by downloading the source files from S3.

This verifies that the pipeline is using the S3-preplaced source files rather than directly depending on the original notebook-generated `src/` files.

In [10]:
import shutil

local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py -> pipeline_src/train.py


Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded files:
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/train.py


## 2. Define the SageMaker Pipeline

Four SageMaker Pipeline steps:

1. **ProcessingStep** — runs `preprocess.py`, outputs train/test CSVs and `preprocessor.joblib` to S3.
2. **TrainingStep** — runs `train.py`, trains the model, loads `preprocessor.joblib`, and saves a `model.joblib` deployment bundle.
3. **ConditionStep** — checks AUC >= threshold before allowing registration.
4. **ModelStep** — registers the model in SageMaker Model Registry (`PendingManualApproval`).

MLflow is not inside the pipeline. A later notebook section logs the completed SageMaker run into the team SageMaker MLflow App experiment.


In [11]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

pipeline_session = PipelineSession()

# Pipeline parameters — can be overridden at execution time
p_n_est    = ParameterInteger(name='NEstimators',    default_value=200)  # matches rf_candidate_05
p_depth    = ParameterInteger(name='MaxDepth',       default_value=8)   # matches rf_candidate_05
p_samples  = ParameterInteger(name='MinSamplesLeaf', default_value=2)   # matches rf_candidate_05
p_gate     = ParameterFloat(  name='QualityGateAUC', default_value=QUALITY_GATE_AUC)

print('Pipeline parameters defined.')


Pipeline parameters defined.


In [12]:
# Step 1: ProcessingStep
processor = SKLearnProcessor(
    framework_version='1.2-1', instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1, role=role, sagemaker_session=pipeline_session,
    base_job_name=f'iti113-{TEAM_ID}-{STUDENT_ID}-process')

step_process = ProcessingStep(
    name='PreprocessData',
    processor=processor,
    inputs=[ProcessingInput(source=RAW_DATA_URI,
                            destination='/opt/ml/processing/input')],  # lands at .../input/crypto_scam_dataset.csv
    outputs=[ProcessingOutput(output_name='processed',
                              source='/opt/ml/processing/output',
                              destination=f'{PIPELINE_ROOT}/processed')],
    code=f'{LOCAL_PIPELINE_SRC}/preprocess.py',
    job_arguments=['--test-size','0.2','--random-state','42']
)
print('Step 1 (ProcessingStep) defined.')


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 1 (ProcessingStep) defined.


In [13]:
# Step 2: TrainingStep
# The training container receives no Databricks host, token, or MLflow dependency.
# It only trains the model and prints metrics for SageMaker to capture.
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "n-estimators": p_n_est,
        "max-depth": p_depth,
        "min-samples-leaf": p_samples,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "sagemaker_pipeline_run",
    },
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    metric_definitions=[
        {"Name": "test_auc_roc", "Regex": "Test AUC-ROC: ([0-9\\.]+)"},
        {"Name": "test_accuracy", "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_f1", "Regex": "test_f1: ([0-9\\.]+)"},
    ],
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        # ADAPTED: no content_type -- Processing output is a mix of .npz (TF-IDF)
        # and .csv (engineered features/labels), not a single CSV like the tutor's.
        # train.py reads each file directly, so this hint isn't needed.
        "train": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
        "test": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
    },
)

print("Step 2 (TrainingStep) defined.")


/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 2 (TrainingStep) defined.


In [14]:
# Step 3: ModelStep — register in SageMaker Model Registry
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_PIPELINE_SRC
)
step_register = ModelStep(
    name='RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('Step 3 (ModelStep) defined.')


/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Step 3 (ModelStep) defined.


In [15]:
# Step 4: ConditionStep — gate on SageMaker-captured test AUC
#
# The training script prints:
#     Test AUC-ROC: 0.xxxx
# and the estimator metric_definitions capture this as "test_auc_roc".
# This avoids relying on Databricks Model Registry or a separate evaluation file.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
    right=p_gate
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("Step 4 (ConditionStep) defined.")


Step 4 (ConditionStep) defined.


In [16]:
# Assemble and upsert the pipeline
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[p_n_est, p_depth, p_samples, p_gate],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session
)
pipeline.upsert(role_arn=role)
print(f'Pipeline "{PIPELINE_NAME}" upserted.')
print('View in SageMaker Studio: left sidebar -> Pipelines')


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team03-crypto-scam-detector" upserted.
View in SageMaker Studio: left sidebar -> Pipelines


## 3. Execute the Pipeline

In [17]:
execution = pipeline.start(parameters={
    'NEstimators':200, 'MaxDepth':8, 'MinSamplesLeaf':2, 'QualityGateAUC':0.85
})
print(f'Execution ARN: {execution.arn}')
print('Monitoring step status below. Takes ~10-15 minutes.')

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector/execution/b7s4pss27ddv
Monitoring step status below. Takes ~10-15 minutes.


In [18]:
import time

prev = {}

while True:

    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]

    steps = execution.list_steps()

    # Compatible with both old and new SageMaker SDKs
    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])

    for step in steps:
        n = step["StepName"]
        s = step["StepStatus"]

        if prev.get(n) != s:
            print(f"{n:<20} {s}")
            prev[n] = s

    if status in ("Succeeded", "Failed", "Stopped"):
        print(f"\nPipeline Status: {status}")
        break

    time.sleep(30)



PreprocessData       Executing


TrainModel           Executing
PreprocessData       Succeeded


RegisterModel-RepackModel-0 Starting
AUCQualityGate       Succeeded
TrainModel           Succeeded


RegisterModel-RepackModel-0 Executing


RegisterModel-RegisterModel Succeeded
RegisterModel-RepackModel-0 Succeeded

Pipeline Status: Succeeded


## 4. Log the Completed SageMaker Run to the SageMaker MLflow App

Run this section only after the pipeline has succeeded.

This notebook-side step reads the completed **SageMaker Training Job** to obtain:
- captured quality metrics,
- training hyperparameters,
- training-job name and pipeline execution ARN,
- SageMaker model-artifact S3 URI.

It then logs these as an MLflow run in the team experiment hosted by the **SageMaker Serverless MLflow App** created in Notebook 01. No Databricks host or token is required.


In [19]:
# Run only after the execution-monitoring cell reports "Pipeline Succeeded".
# In SageMaker SDK 2.257.3, execution.list_steps() returns a Python list.
# The compatibility helper also supports SDK versions that return a dictionary.
import mlflow


def get_pipeline_steps(execution):
    response = execution.list_steps()
    if isinstance(response, list):
        return response
    return response.get("PipelineExecutionSteps", [])


if execution.describe()["PipelineExecutionStatus"] != "Succeeded":
    raise RuntimeError(
        "The SageMaker Pipeline has not succeeded. "
        "Resolve pipeline failures before logging to MLflow."
    )

steps = get_pipeline_steps(execution)

print("Pipeline steps:")
for step in steps:
    print(f"  {step['StepName']}: {step['StepStatus']}")

train_step_info = next(
    (
        step for step in steps
        if step["StepName"] == "TrainModel"
        and step["StepStatus"] == "Succeeded"
    ),
    None
)

if train_step_info is None:
    raise RuntimeError(
        "A successful TrainModel step was not found in this pipeline execution."
    )

training_job_arn = train_step_info["Metadata"]["TrainingJob"]["Arn"]
training_job_name = training_job_arn.rsplit("/", 1)[-1]

sm_client = boto3.client("sagemaker", region_name=region)
training_job = sm_client.describe_training_job(
    TrainingJobName=training_job_name
)

# SageMaker captures the metrics printed by train.py through metric_definitions.
captured_metrics = {
    item["MetricName"]: float(item["Value"])
    for item in training_job.get("FinalMetricDataList", [])
    if item["MetricName"] in {"test_auc_roc", "test_accuracy", "test_f1"}
}

if not captured_metrics:
    raise RuntimeError(
        "No captured SageMaker metrics were found. "
        "Check train.py output and estimator.metric_definitions."
    )

model_artifact_s3_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]
training_hyperparameters = training_job.get("HyperParameters", {})

print("Training job:", training_job_name)
print("Model artefact:", model_artifact_s3_uri)
print("Captured metrics:", captured_metrics)

# Log to SageMaker Serverless MLflow App.
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow_run_name = (
    f"{TEAM_ID}_{STUDENT_ID}_sagemaker_pipeline_"
    f"{int(time.time())}"
)

with mlflow.start_run(run_name=mlflow_run_name) as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "crypto-scam-detector",
        "execution_environment": "aws_sagemaker_pipeline",
        "tracking_backend": "sagemaker_mlflow_app",
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "sagemaker_model_artifact_s3_uri": model_artifact_s3_uri,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
    })

    # Hyperparameters arrive from SageMaker as strings, which are valid MLflow params.
    mlflow.log_params(training_hyperparameters)
    mlflow.log_metrics(captured_metrics)

    # Store a small, portable traceability record as an MLflow artefact.
    run_summary = {
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "training_job_arn": training_job_arn,
        "model_artifact_s3_uri": model_artifact_s3_uri,
        "metrics": captured_metrics,
        "hyperparameters": training_hyperparameters,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "semester": SEMESTER,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
        "tracking_backend": "sagemaker_mlflow_app",
    }

    summary_file = "sagemaker_pipeline_run_summary.json"
    with open(summary_file, "w") as f:
        json.dump(run_summary, f, indent=2)

    mlflow.log_artifact(
        summary_file,
        artifact_path="sagemaker_pipeline"
    )

    mlflow_run_id = run.info.run_id
    mlflow_experiment_id = run.info.experiment_id

print("SageMaker MLflow App logging completed.")
print("MLflow run ID:", mlflow_run_id)
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", mlflow_experiment_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=mlflow_experiment_id,
    run_id=mlflow_run_id
)


Pipeline steps:
  RegisterModel-RegisterModel: Succeeded
  RegisterModel-RepackModel-0: Succeeded
  AUCQualityGate: Succeeded
  TrainModel: Succeeded
  PreprocessData: Succeeded


Training job: pipelines-b7s4pss27ddv-TrainModel-G6XnH3sgHf
Model artefact: s3://sagemaker-ap-southeast-1-044528205969/pipelines-b7s4pss27ddv-TrainModel-G6XnH3sgHf/output/model.tar.gz
Captured metrics: {'test_auc_roc': 1.0, 'test_accuracy': 0.9994999766349792, 'test_f1': 0.9994999766349792}


🏃 View run team03_s301_sagemaker_pipeline_1785567057 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/ed52087946624ea19ad401cec6366ba7
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App logging completed.
MLflow run ID: ed52087946624ea19ad401cec6366ba7
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
Experiment: ITI113/team03/Experiment1
Experiment ID: 1

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-J5AYUG4AJHVW.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IjU1UzJIUCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDYybFpDc3cvTHM5ZTNtdmZ2dk94bW9icyttUG9ReFdTWUhJaUE1Mi9CeFlBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFM2VWSTRNbkpYYWxaSWNFdEdXV2t4ZFVrMFdpdGFURFUwWW5GWlFYUlBPVmx6VUROQk5qWkJURmx3V0hGVlVEaDBUemd4YUhaVk5USjVRelZFVDFscVFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFkdXQ5cjFYeTc4ZkVJb0p0V09oTU5VQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4UUd3eWx6K3BGbndsamdpTUNBUkNBT3hIVkZiVExzUnpGZFRIcW0zMzM4eTVyODJuaEkzN25xaTdFVENLcHVndy9JQ05jaUNYQ08xZUF3MWtiR28xd3ptN0JUeFVPd3F0cmUxQldBZ0FBRUFDbzF3VHh3RVNUQWJzbG44Y1VSemVwSzh1Rko0bG5FOG9pUHlER1l6WjZMMDE1S3lReG5BUl


Presigned MLflow run URL:
https://app-J5AYUG4AJHVW.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkhVR1dRWCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHYvZ2FzekJMMVl2YmxLUGFZTWU4MFpmdnJJSmRLdnZvNjZzQWg0OXZQK3dBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGMVptcFFUazk2TUVvMWNXeFhReXRHWkZONWVIUktaSEpLUTJ3MlVqbE1lR05MVjFZeVJscFZiMWN3Tldwb1JrdHViMDA0VjJFeGNHZFJhbEpQUjNCTlFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFhQjgyb0RTajZQSTZNWTc0amJIbktNQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6OTFRQTdYY3JLZ09lSE56RUNBUkNBTytkZVlCc0l6VmZCRlJkY0MreFhUN2FKUXZxd1ZmWWhHM3B4eW11dXp1bEExSm1JOVJlOTdpOE5EZkJHYkRHR2xCeVR2YjFsb2FSZ2pHbVFBZ0FBRUFCY2Y2Wlp3S0pQK2EvSm1HNlY0WUt2cmpjSEpJdi9jeWZLVVhGTEMyaUN1ZVpvVnBYRzNwSGpvQ0gx

## 5. Deploy Serverless Endpoint

After the pipeline succeeds, the model sits in Model Registry with `PendingManualApproval`.
We approve it here, then deploy as a **Serverless Endpoint** — cost is near-zero when idle.


In [20]:
sm = boto3.client('sagemaker')

# Get the latest registered model package
pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy='CreationTime', SortOrder='Descending', MaxResults=1
)['ModelPackageSummaryList']

if not pkgs:
    print('No model packages found. Check the pipeline completed the Register step.')
else:
    pkg_arn = pkgs[0]['ModelPackageArn']
    print(f'Model package : {pkg_arn}')
    print(f'Status        : {pkgs[0]["ModelApprovalStatus"]}')


Model package : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/1
Status        : PendingManualApproval


In [21]:
# Approve the model
sm.update_model_package(ModelPackageArn=pkg_arn, ModelApprovalStatus='Approved')
print(f'Approved: {pkg_arn}')


Approved: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/1


In [22]:
from sagemaker import ModelPackage
from sagemaker.serverless import ServerlessInferenceConfig

deployable = ModelPackage(
    role=role, model_package_arn=pkg_arn, sagemaker_session=sagemaker.Session())

serverless_cfg = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=5)

print(f'Deploying serverless endpoint: {ENDPOINT_NAME}')
print('This takes 3-5 minutes...')
predictor = deployable.deploy(
    serverless_inference_config=serverless_cfg,
    endpoint_name=ENDPOINT_NAME
)
print(f'Endpoint ready: {ENDPOINT_NAME}')
print('Cost: ~$0 idle. Charged per invocation only.')


/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: ModelPackage is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Deploying serverless endpoint: iti113-team03-crypto-scam-detector
This takes 3-5 minutes...


INFO:sagemaker:Creating model with name: team03-CryptoScamDetector-2026-08-01-06-50-59-528


INFO:sagemaker:Creating endpoint-config with name iti113-team03-crypto-scam-detector


INFO:sagemaker:Creating endpoint with name iti113-team03-crypto-scam-detector


...

-

-

-

!

Endpoint ready: iti113-team03-crypto-scam-detector
Cost: ~$0 idle. Charged per invocation only.


## 6. Test the Live Endpoint

The deployed endpoint now accepts **raw 13-feature JSON input**. The endpoint's `inference.py` performs the deployment-time preprocessing in memory using the fitted preprocessing artefacts saved in `model.joblib`.

No separate SageMaker Processing job is started during live inference.

Expected raw input fields:

```text
age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal
```


In [24]:
import json

rt = boto3.client('sagemaker-runtime', region_name=region)

# Scam-style message: urgency + guaranteed-return language + payment request.
scam_message = {
    "text": (
        "URGENT: Your wallet has been selected for a guaranteed 100% profit "
        "airdrop! Deposit 500 USDT to your wallet address within 1 hour to "
        "claim now. Contact us on Telegram immediately, don't miss out!"
    )
}

resp = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps(scam_message)
)

result = json.loads(resp['Body'].read())[0]
print('SCAM-STYLE MESSAGE (urgency + guaranteed return + payment request)')
print(f'  Prediction  : {result["label"]}')
print(f'  Probability : {result["probability"]:.1%}')

SCAM-STYLE MESSAGE (urgency + guaranteed return + payment request)
  Prediction  : Scam
  Probability : 75.8%


In [25]:
# Legit-style message for comparison.
legit_message = {
    "text": (
        "Been dollar-cost averaging into ETH for about a year now, curious "
        "what everyone's thoughts are on the current market conditions."
    )
}

resp2 = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=json.dumps(legit_message)
)

result2 = json.loads(resp2['Body'].read())[0]
print('LEGIT-STYLE MESSAGE (casual discussion, no urgency/payment indicators)')
print(f'  Prediction  : {result2["label"]}')
print(f'  Probability : {result2["probability"]:.1%}')

LEGIT-STYLE MESSAGE (casual discussion, no urgency/payment indicators)
  Prediction  : Legitimate
  Probability : 21.8%


#### Delete endpoint when no longer needed

The endpoint is serverless, but it should still be cleaned up after class testing to avoid clutter and accidental usage. The cleanup code below is intentionally commented out. Only uncomment it when you are sure the endpoint is no longer needed.


In [ ]:
# To delete endpoint when no longer needed, uncomment the lines below:
# sm_cleanup = boto3.client('sagemaker', region_name=REGION)
# sm_cleanup.delete_endpoint(EndpointName=ENDPOINT_NAME)
# print(f'Endpoint deleted: {ENDPOINT_NAME}')


In [26]:
print('=' * 55)
print('NOTEBOOK 03 COMPLETE')
print('=' * 55)
print(f'Pipeline : {PIPELINE_NAME}')
print(f'MLflow   : {MLFLOW_EXPERIMENT_NAME} on SageMaker MLflow App')
print(f'MLflow App ARN : {MLFLOW_APP_ARN}')
print(f'SageMaker Registry : {MODEL_PACKAGE_GROUP}')
print(f'Endpoint : {ENDPOINT_NAME} (Serverless)')
print()
print('Next: wire pages/1_Scam_Detector.py in the Streamlit app to this')
print('endpoint via boto3 sagemaker-runtime.invoke_endpoint(), replacing the')
print('rule-based placeholder logic. See pages/2_About_the_Model.py too.')

NOTEBOOK 03 COMPLETE
Pipeline : iti113-team03-crypto-scam-detector
MLflow   : ITI113/team03/Experiment1 on SageMaker MLflow App
MLflow App ARN : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
SageMaker Registry : team03-CryptoScamDetector
Endpoint : iti113-team03-crypto-scam-detector (Serverless)

Next: wire pages/1_Scam_Detector.py in the Streamlit app to this
endpoint via boto3 sagemaker-runtime.invoke_endpoint(), replacing the
rule-based placeholder logic. See pages/2_About_the_Model.py too.


## 6. GitHub Actions CI/CD [SKIP THIS]

Save the workflow below as `.github/workflows/run_pipeline.yml` in your repository.  
It will re-run the pipeline automatically on every push to `main`.

**Setup:** GitHub → Settings → Secrets → Actions → add `AWS_ROLE_ARN` and `SAGEMAKER_BUCKET`. MLflow App permissions are not needed by the SageMaker pipeline itself unless the CI job also performs post-pipeline MLflow logging.


In [27]:
workflow_yaml = '''
name: Run SageMaker MLOps Pipeline

on:
  push:
    branches: [ main ]
    paths: [ 'src/**', 'pipeline/**' ]

permissions:
  id-token: write
  contents: read

jobs:
  run-pipeline:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.10' }
      - run: pip install sagemaker boto3
      - uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: ${{ secrets.AWS_ROLE_ARN }}
          aws-region: ap-southeast-1
      - env:
          SAGEMAKER_BUCKET: ${{ secrets.SAGEMAKER_BUCKET }}
        run: python pipeline/run_pipeline.py --pipeline-name iti113-T01-heart-disease --wait
'''
print(workflow_yaml)
# Optionally save to disk:
# os.makedirs('.github/workflows', exist_ok=True)
# open('.github/workflows/run_pipeline.yml','w').write(workflow_yaml.strip())



name: Run SageMaker MLOps Pipeline

on:
  push:
    branches: [ main ]
    paths: [ 'src/**', 'pipeline/**' ]

permissions:
  id-token: write
  contents: read

jobs:
  run-pipeline:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.10' }
      - run: pip install sagemaker boto3
      - uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: ${{ secrets.AWS_ROLE_ARN }}
          aws-region: ap-southeast-1
      - env:
          SAGEMAKER_BUCKET: ${{ secrets.SAGEMAKER_BUCKET }}
        run: python pipeline/run_pipeline.py --pipeline-name iti113-T01-heart-disease --wait



---
## Checklist before Notebook 04

- [ ] `src/preprocess.py`, `train.py`, `inference.py` written and reviewed
- [ ] `preprocess.py` saves `preprocessor.joblib`
- [ ] `train.py` saves `model.joblib` containing the trained model and preprocessing artefacts
- [ ] `inference.py` accepts raw 13-feature JSON input and performs deployment-time preprocessing
- [ ] Pipeline upserted (visible in SageMaker Studio under Pipelines)
- [ ] Pipeline execution completed (all steps green)
- [ ] Post-pipeline MLflow run visible in the team SageMaker MLflow App experiment
- [ ] Model registered in SageMaker Model Registry after passing AUC gate
- [ ] Serverless Endpoint deployed and responding to raw JSON test calls
- [ ] Both high-risk and low-risk raw test profiles return sensible predictions


### Below Code Gets the model from endpoint (make sure endpoint is not deleted)

In [28]:
import os
import json
import boto3
from urllib.parse import urlparse
from pathlib import Path
from botocore.exceptions import ClientError


# ============================================================
# CONFIGURATION
# ============================================================

REGION = "ap-southeast-1"

# Your deployed serverless endpoint name
ENDPOINT_NAME = ENDPOINT_NAME #"REPLACE_WITH_YOUR_ENDPOINT_NAME"

# Your class/team bucket and desired destination folder
DESTINATION_BUCKET = "nyp-26s1-iti113" # "REPLACE_WITH_YOUR_CLASS_BUCKET"

# Recommended destination naming
DESTINATION_KEY = (
    f"iti113/{TEAM_ID}/models/"
    f"{PROJECT_NAME}/"
    "model-package-v2/"
    "model.tar.gz"
)

# Optional local notebook download folder
LOCAL_DOWNLOAD_DIR = Path("downloaded_models")


# ============================================================
# AWS CLIENTS
# ============================================================

sm_client = boto3.client("sagemaker", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)
s3_resource = boto3.resource("s3", region_name=REGION)


# ============================================================
# HELPER: Parse an S3 URI
# ============================================================

def parse_s3_uri(s3_uri: str):
    """
    Convert:
        s3://bucket-name/path/to/object
    into:
        bucket-name, path/to/object
    """
    parsed = urlparse(s3_uri)

    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")

    return parsed.netloc, parsed.path.lstrip("/")


# ============================================================
# STEP 1: Endpoint -> Endpoint Config -> SageMaker Model
# ============================================================

endpoint_desc = sm_client.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)

endpoint_config_name = endpoint_desc["EndpointConfigName"]

endpoint_config_desc = sm_client.describe_endpoint_config(
    EndpointConfigName=endpoint_config_name
)

production_variants = endpoint_config_desc["ProductionVariants"]

if not production_variants:
    raise ValueError("No production variants found in endpoint configuration.")

model_name = production_variants[0]["ModelName"]

model_desc = sm_client.describe_model(
    ModelName=model_name
)

print("Endpoint name:", ENDPOINT_NAME)
print("Endpoint status:", endpoint_desc["EndpointStatus"])
print("Endpoint configuration:", endpoint_config_name)
print("SageMaker model:", model_name)


# ============================================================
# STEP 2: SageMaker Model -> Model Package
# ============================================================

containers = model_desc.get("Containers", [])

if not containers:
    raise ValueError(
        "No Containers found in SageMaker model definition. "
        "Expected a model created from a Model Package."
    )

model_package_arn = containers[0].get("ModelPackageName")

if not model_package_arn:
    raise ValueError(
        "This model does not contain ModelPackageName. "
        "Inspect model_desc manually for a direct ModelDataUrl."
    )

print("Model package ARN:", model_package_arn)

package_desc = sm_client.describe_model_package(
    ModelPackageName=model_package_arn
)

package_containers = package_desc["InferenceSpecification"]["Containers"]

if not package_containers:
    raise ValueError("No inference containers found in model package.")

model_s3_uri = package_containers[0].get("ModelDataUrl")

if not model_s3_uri:
    raise ValueError(
        "ModelDataUrl not found in model package inference container.\n"
        + json.dumps(package_containers[0], indent=2, default=str)
    )

print("\nOriginal model artefact S3 URI:")
print(model_s3_uri)


# ============================================================
# STEP 3: Verify source object exists
# ============================================================

source_bucket, source_key = parse_s3_uri(model_s3_uri)

source_metadata = s3_client.head_object(
    Bucket=source_bucket,
    Key=source_key
)

source_size_mb = source_metadata["ContentLength"] / (1024 * 1024)

print("\nSource bucket:", source_bucket)
print("Source key:", source_key)
print(f"Source model size: {source_size_mb:.2f} MB")


# ============================================================
# STEP 4: Download locally to the Studio notebook environment
# ============================================================

LOCAL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

local_model_file = LOCAL_DOWNLOAD_DIR / "model.tar.gz"

print("\nDownloading model locally...")
s3_client.download_file(
    source_bucket,
    source_key,
    str(local_model_file)
)

print("Local file saved to:")
print(local_model_file.resolve())

print(f"Local file size: {local_model_file.stat().st_size / (1024 * 1024):.2f} MB")


# ============================================================
# STEP 5: Copy the model directly into your team S3 prefix
# ============================================================

copy_source = {
    "Bucket": source_bucket,
    "Key": source_key
}

print("\nCopying model into team S3 folder...")

s3_resource.meta.client.copy(
    CopySource=copy_source,
    Bucket=DESTINATION_BUCKET,
    Key=DESTINATION_KEY
)

destination_s3_uri = f"s3://{DESTINATION_BUCKET}/{DESTINATION_KEY}"

print("\nCopy completed successfully.")
print("Destination model artefact:")
print(destination_s3_uri)


# ============================================================
# STEP 6: Verify destination object
# ============================================================

destination_metadata = s3_client.head_object(
    Bucket=DESTINATION_BUCKET,
    Key=DESTINATION_KEY
)

print("\nDestination verification:")
print("Destination size (MB):", round(
    destination_metadata["ContentLength"] / (1024 * 1024),
    2
))
print("Last modified:", destination_metadata["LastModified"])
print("ETag:", destination_metadata["ETag"])


Endpoint name: iti113-team03-crypto-scam-detector
Endpoint status: InService
Endpoint configuration: iti113-team03-crypto-scam-detector
SageMaker model: team03-CryptoScamDetector-2026-08-01-06-50-59-528
Model package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/1

Original model artefact S3 URI:
s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-08-01-06-39-51-348/pipelines-b7s4pss27ddv-RegisterModel-Repack-Oq1u4PRhqT/output/model.tar.gz

Source bucket: sagemaker-ap-southeast-1-044528205969
Source key: sagemaker-scikit-learn-2026-08-01-06-39-51-348/pipelines-b7s4pss27ddv-RegisterModel-Repack-Oq1u4PRhqT/output/model.tar.gz
Source model size: 1.40 MB



Local file saved to:
/home/sagemaker-user/team03/downloaded_models/model.tar.gz
Local file size: 1.40 MB

Copying model into team S3 folder...



Copy completed successfully.
Destination model artefact:
s3://nyp-26s1-iti113/iti113/team03/models/crypto-scam-detector/model-package-v2/model.tar.gz

Destination verification:
Destination size (MB): 1.4
Last modified: 2026-08-01 06:56:14+00:00
ETag: "e4587747f0f9e9a6edd87e81e94f599d"
